Для сглаживания стыков полиномиальных моделей в твоем тренде (1440 значений), у тебя есть несколько эффективных подходов:

---

## 🔧 Подходы к сглаживанию стыков:

### ✅ Подход 1: Плавное соединение полиномов через взвешенные средние

* На стыках каждой пары частей (например, конец первой и начало второй) ты можешь использовать **взвешенное среднее**:

  $$
  \text{Сглаженный\_значение} = (1 - w) \times \text{Полином1} + w \times \text{Полином2}
  $$

  где $w$ — вес, изменяющийся от 0 до 1 на интервале стыка (например, 10–20 значений).

* Это создаст плавный переход без резких скачков.

---

### ✅ Подход 2: Сплайновая регрессия (Cubic Spline)

* Вместо полиномов можно использовать **сплайны**:

  * Раздели тренд на те же 3 части.
  * Построй кубический сплайн (или сплайн более высокой степени) с ограничением на непрерывность и гладкость (производная на границах).
* Это позволяет плавно соединять части без стыков.

---

### ✅ Подход 3: Сглаживание через скользящее среднее

* После построения полиномиальных регрессий сгладь стыки через **скользящее среднее** (на 10-20 точек).
* Это уберет резкие переходы и сделает кривую более плавной.

---

### ✅ Подход 4: Локальная регрессия (LOESS)

* Используй LOESS/LOWESS (локальную регрессию), которая строит гладкую линию без явных стыков.
* Она автоматически адаптируется к локальным особенностям данных и исключает проблему стыков.

---

## 🔧 Какой из подходов тебе подходит больше?

Хотел бы, чтобы я показал тебе, как реализовать взвешенное сглаживание (подход 1) или сплайновую регрессию (подход 2) на твоих данных в Python?


In [ ]:
# 4. Плавное сглаживание тренда (взвешенное сглаживание на стыках полиномов)
def smooth_trend_weighted_segments(trend, segments, transition=20):
    smoothed = trend.copy()

    for i in range(len(segments) - 1):
        start = segments[i][1] - transition
        end = segments[i + 1][0] + transition

        for j in range(transition):
            weight = j / transition
            smoothed[start + j] = (1 - weight) * trend[start + j] + weight * trend[end - transition + j]
            smoothed[end - j - 1] = (1 - weight) * trend[end - j - 1] + weight * trend[start + transition - j - 1]

    return smoothed

### 📌 Как работает этот подход:

* Функция `smooth_trend_weighted_segments` принимает:

  * `trend` — твой массив значений тренда.
  * `segments` — список кортежей, где каждый кортеж — это `(start, end)` индексы сегмента (т.е. начало и конец каждого полинома).
  * `transition` — количество точек на стыках, которые будут сглажены.

* Сглаживание выполняется на стыках полиномов:

  * В начале стыка мы создаем плавный переход, где веса линейно изменяются от 0 до 1.
  * В конце стыка — обратный переход от 1 до 0.
  * Это устраняет резкие скачки между сегментами.

---

### 📊 Как использовать:

* Предположим, у тебя три сегмента тренда:

  ```python
  trend = np.array([...])  # Твой тренд (например, 1440 значений)
  segments = [(0, 480), (480, 960), (960, 1440)]  # Индексы начала и конца каждого полинома

  smoothed_trend = smooth_trend_weighted_segments(trend, segments, transition=20)
  ```

* Хочешь, я покажу, как это визуализировать и сравнить исходный тренд и сглаженный? 🙂


### **1. Использование KNN для сглаживания тренда**

**Проблема:** На хвостах тренда остается большое отклонение, особенно если тренд начинается с пика.

**Решение:** 
- Увеличение количества соседей помогает сгладить тренд, но может привести к потере деталей на хвостах.
- Можно использовать **оконный метод KNN**, который ограничивает область поиска соседей только в пределах окна вокруг текущей точки.

#### **Пример кода с использованием оконного метода KNN:**

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor

# Пример данных
np.random.seed(0)
dates = pd.date_range(start='2023-01-01', periods=1440, freq='T')  # Пример за один день
data = pd.DataFrame({
    'value': np.random.rand(1440) * 100  # Случайные значения от 0 до 100
})

# Добавим несколько пиков для демонстрации
peak_indices = [100, 250, 400, 600, 800, 1000, 1200]
data.loc[peak_indices, 'value'] += 100  # Добавляем пиковые значения

# Визуализация исходных данных
plt.figure(figsize=(14, 6))
plt.plot(data.index, data['value'], label='Исходные данные', color='blue')
plt.scatter(data.index[peak_indices], data['value'][peak_indices], color='red', label='Пики')
plt.title('Исходные данные с пиками')
plt.xlabel('Дата')
plt.ylabel('Значение')
plt.legend()
plt.show()

# Оконный метод KNN
def knn_smoothing(data, window_size=100, n_neighbors=50):
    smoothed_data = data.copy()
    knn = KNeighborsRegressor(n_neighbors=n_neighbors)
    
    for i in range(len(data)):
        start_index = max(0, i - window_size // 2)
        end_index = min(len(data), i + window_size // 2 + 1)
        
        X_train = np.arange(start_index, end_index).reshape(-1, 1)
        y_train = data.iloc[start_index:end_index]['value']
        
        knn.fit(X_train, y_train)
        smoothed_data.at[i, 'value'] = knn.predict(np.array([[i]]))[0]
    
    return smoothed_data

# Применение оконного метода KNN
smoothed_data_knn = knn_smoothing(data, window_size=100, n_neighbors=50)

# Визуализация результатов
plt.figure(figsize=(14, 6))
plt.plot(data.index, data['value'], label='Исходные данные', color='blue', alpha=0.5)
plt.scatter(data.index[peak_indices], data['value'][peak_indices], color='red', label='Пики')
plt.plot(smoothed_data_knn.index, smoothed_data_knn['value'], label='Сглаженный тренд (KNN)', color='green')
plt.title('Сглаживание тренда с использованием KNN')
plt.xlabel('Дата')
plt.ylabel('Значение')
plt.legend()
plt.show()
```

#### **Объяснение:**
- **Оконный метод KNN**: Ограничивает область поиска соседей только в пределах окна вокруг текущей точки. Это помогает избежать большого отклонения на хвостах тренда.

### **2. Полиномиальная регрессия с устранением стыков**

**Проблема:** На стыках частей видны стыки, так как полиномиальные модели не совпадают точно на стыках.

**Решение:** 
- Используй **сплайн-интерполяцию** или **полиномиальную регрессию с непрерывностью на стыках**.
- **Сплайн-интерполяция** обеспечивает непрерывность и гладкость на стыках.

#### **Пример кода с использованием сплайн-интерполяции:**

```python
from scipy.interpolate import UnivariateSpline

# Пример данных
np.random.seed(0)
dates = pd.date_range(start='2023-01-01', periods=1440, freq='T')  # Пример за один день
data = pd.DataFrame({
    'value': np.random.rand(1440) * 100  # Случайные значения от 0 до 100
})

# Добавим несколько пиков для демонстрации
peak_indices = [100, 250, 400, 600, 800, 1000, 1200]
data.loc[peak_indices, 'value'] += 100  # Добавляем пиковые значения

# Визуализация исходных данных
plt.figure(figsize=(14, 6))
plt.plot(data.index, data['value'], label='Исходные данные', color='blue')
plt.scatter(data.index[peak_indices], data['value'][peak_indices], color='red', label='Пики')
plt.title('Исходные данные с пиками')
plt.xlabel('Дата')
plt.ylabel('Значение')
plt.legend()
plt.show()

# Сплайн-интерполяция
def spline_smoothing(data, degree=3, s=100):
    x = np.arange(len(data))
    y = data['value']
    spl = UnivariateSpline(x, y, k=degree, s=s)
    smoothed_y = spl(x)
    return pd.Series(smoothed_y, index=data.index)

# Применение сплайн-интерполяции
smoothed_data_spline = spline_smoothing(data, degree=3, s=100)

# Визуализация результатов
plt.figure(figsize=(14, 6))
plt.plot(data.index, data['value'], label='Исходные данные', color='blue', alpha=0.5)
plt.scatter(data.index[peak_indices], data['value'][peak_indices], color='red', label='Пики')
plt.plot(smoothed_data_spline.index, smoothed_data_spline, label='Сглаженный тренд (Сплайн)', color='green')
plt.title('Сглаживание тренда с использованием сплайн-интерполяции')
plt.xlabel('Дата')
plt.ylabel('Значение')
plt.legend()
plt.show()
```

#### **Объяснение:**
- **Сплайн-интерполяция**: Использует кусочно-полиномиальные функции, которые обеспечивают непрерывность и гладкость на стыках. Параметр `s` контролирует степень сглаживания.

### **Альтернативные методы сглаживания**

#### **a. Экспоненциальное сглаживание**

Экспоненциальное сглаживание также может быть эффективным способом сглаживания тренда.

```python
# Экспоненциальное сглаживание
def exponential_smoothing(data, alpha=0.1):
    smoothed_data = data.copy()
    smoothed_data['value'] = smoothed_data['value'].ewm(alpha=alpha, adjust=False).mean()
    return smoothed_data

# Применение экспоненциального сглаживания
smoothed_data_exp = exponential_smoothing(data, alpha=0.1)

# Визуализация результатов
plt.figure(figsize=(14, 6))
plt.plot(data.index, data['value'], label='Исходные данные', color='blue', alpha=0.5)
plt.scatter(data.index[peak_indices], data['value'][peak_indices], color='red', label='Пики')
plt.plot(smoothed_data_exp.index, smoothed_data_exp['value'], label='Сглаженный тренд (Экспоненциальное сглаживание)', color='green')
plt.title('Сглаживание тренда с использованием экспоненциального сглаживания')
plt.xlabel('Дата')
plt.ylabel('Значение')
plt.legend()
plt.show()
```

#### **b. Медианное сглаживание**

Медианное сглаживание также может быть полезным, особенно если данные содержат выбросы.

```python
# Медианное сглаживание
def median_smoothing(data, window_size=10):
    smoothed_data = data.copy()
    smoothed_data['value'] = smoothed_data['value'].rolling(window=window_size, center=True).median()
    smoothed_data['value'].iloc[:window_size//2] = smoothed_data['value'].iloc[window_size//2]
    smoothed_data['value'].iloc[-window_size//2:] = smoothed_data['value'].iloc[-window_size//2-1]
    return smoothed_data

# Применение медианного сглаживания
smoothed_data_median = median_smoothing(data, window_size=10)

# Визуализация результатов
plt.figure(figsize=(14, 6))
plt.plot(data.index, data['value'], label='Исходные данные', color='blue', alpha=0.5)
plt.scatter(data.index[peak_indices], data['value'][peak_indices], color='red', label='Пики')
plt.plot(smoothed_data_median.index, smoothed_data_median['value'], label='Сглаженный тренд (Медианное сглаживание)', color='green')
plt.title('Сглаживание тренда с использованием медианного сглаживания')
plt.xlabel('Дата')
plt.ylabel('Значение')
plt.legend()
plt.show()
```

### **Рекомендации**

1. **Оконный метод KNN:**
   - Используй оконный метод KNN для сглаживания тренда, чтобы избежать больших отклонений на хвостах.
   - Настрой параметры `window_size` и `n_neighbors` для достижения оптимального сглаживания.

2. **Сплайн-интерполяция:**
   - Используй сплайн-интерполяцию для устранения стыков между частями полиномиальной регрессии.
   - Настрой параметры `degree` и `s` для достижения оптимального сглаживания.

3. **Экспоненциальное сглаживание:**
   - Экспоненциальное сглаживание может быть полезным для сглаживания данных с пиками и выбросами.
   - Настрой параметр `alpha` для достижения оптимального сглаживания.

4. **Медианное сглаживание:**
   - Медианное сглаживание эффективно удаляет выбросы и пиковые значения.
   - Настрой параметр `window_size` для достижения оптимального сглаживания.

### **Заключение**

- **Оконный метод KNN**: Эффективен для сглаживания тренда, особенно на хвостах.
- **Сплайн-интерполяция**: Устраняет стыки между частями полиномиальной регрессии, обеспечивая непрерывность и гладкость.
- **Экспоненциальное сглаживание**: Полезно для данных с пиками и выбросами.
- **Медианное сглаживание**: Эффективно удаляет выбросы и пиковые значения.

Выберите метод, который лучше всего подходит для твоих данных, или комбинируйте несколько методов для достижения наилучшего результата.

Если у тебя есть дополнительные вопросы или нужна помощь с конкретными шагами, дай знать! 😊

In [ ]:
# Сглаживание KNN
# Сглаживание с помощью полинимиальной регрессии.
# Сглаживание с помощью кластерного анализа.

In [ ]:
# Замена выбросов на скользящее среднее
window_size = 5
rolling_mean = series.rolling(window=window_size, center=True).mean()

▌Значения center:

| Значение | Описание | Как влияет? |
| --- | --- | --- |
| center=False (по умолчанию) | Окно «завернуто» вправо — то есть, окно заканчивается на текущем элементе и смотрит в прошлое | Для элемента с индексом i окно охватывает [i - window_size + 1, i] |
| center=True | Окно «центрировано» — то есть, текущее значение — центр окна | Для элемента с индексом i окно охватывает [i - floor(window_size/2), i + ceil(window_size/2) - 1] |

---

▌Почему это важно?

- center=True делает так, что сглаживание (например, скользящее среднее) симметрично относительно текущей точки.
- Это полезно, когда важно, чтобы сглаженные значения были равномерно распределены вокруг текущего времени, а не смещены в прошлое.

---

▌Пример наглядный

Допустим, у вас есть последовательность:
Python

import pandas as pd
s = pd.Series([1, 2, 3, 4, 5, 6, 7])

- Без center (по умолчанию):
s.rolling(window=3).mean()
Результат: [NaN, NaN, 2.0, 3.0, 4.0, 5.0, 6.0]
Средние для окон: [1,2,3], [2,3,4], [3,4,5], [4,5,6], [5,6,7]

- С center=True:
Python

s.rolling(window=3, center=True).mean()
Результат: [NaN, 2.0, 3.0, 4.0, 5.0, 6.0, NaN]
Средние для окон: [1,2,3], [2,3,4], [3,4,5], [4,5,6], [5,6,7]
Но значения смещены так, что среднее центрировано относительно текущего элемента.

---

▌Итог

- center=True — сглаживание симметрично относительно текущего элемента.
- center=False — сглаживание «слева», то есть, окно включает текущий и предыдущие элементы.